# 00 — Data Preprocessing, EDA & Feature Engineering
**Project:** DSAI 305 — XAI Phase 2  
**Model:** CellMorphNet (Xu et al., IEEE JBHI 2025)  
**Dataset:** TCGA-COAD pre-tiled patches + cBioPortal TMB labels

| Section | Criterion | Points |
|---------|-----------|--------|
| §1–§3   | Data Preprocessing | 0.5 |
| §4–§5   | EDA                | 0.5 |
| §6      | Feature Engineering | 0.5 |


## § 1 — Environment Setup


In [1]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Uncomment on first run:
# pip_install("polars", "Pillow", "matplotlib", "seaborn",
#             "scikit-learn", "tqdm", "scikit-image", "numpy", "pandas")


In [2]:
import os, json, warnings, tarfile, io
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
print(f"Polars  {pl.__version__}")
print(f"NumPy   {np.__version__}")


Polars  1.40.1
NumPy   2.4.4


/home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## § 2 — Project Paths


In [3]:
# Auto-detect project root by looking for pyproject.toml
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd()
)

DATA_DIR        = PROJECT_ROOT / "data"
TILES_DIR       = DATA_DIR / "processed" / "tiles"
CBIO_DIR        = DATA_DIR / "cbioportal_tabular_downloads"
CHECKPOINTS_DIR = DATA_DIR / "checkpoints"
SPLITS_DIR      = PROJECT_ROOT / "members" / "youssef" / "experiments" / "CellMorphNet"

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# TMB thresholds (paper: Section III-A)
TMB_HIGH   = 10.0   # >= 10 mut/Mb
TMB_LOW    = 1.1    # < 1.1 mut/Mb
# Binary: High vs Low+Medium
# Ternary: High / Medium / Low

print("Project root:", PROJECT_ROOT)
print("Tiles dir   :", TILES_DIR)
print("cBioPortal  :", CBIO_DIR)


Project root: /home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI
Tiles dir   : /home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/data/processed/tiles
cBioPortal  : /home/youssef_mohammad/projects/DSAI_305_XAI_PROJECT/DSAI_305_XAI/data/cbioportal_tabular_downloads


## § 3 — Load Clinical Data & TMB Labels

We read the cBioPortal archive () for the COAD study,
extracting  which contains .


In [4]:
SKIP_META = 4  # cBioPortal prepends 4 comment lines

def read_cbio_tar(tar_path: Path) -> dict:
    out = {}
    with tarfile.open(str(tar_path), "r:gz") as tar:
        for member in tar.getmembers():
            fname = Path(member.name).name
            if fname not in {"data_clinical_patient.txt", "data_clinical_sample.txt"}:
                continue
            f = tar.extractfile(member)
            raw = f.read().decode("utf-8", errors="replace").splitlines()
            body = "
".join(raw[SKIP_META:]).encode("utf-8")
            key = "patient" if "patient" in fname else "sample"
            out[key] = pl.read_csv(io.BytesIO(body), separator="	",
                                    infer_schema_length=500, ignore_errors=True)
    return out


# Try COAD study first, fall back to any available tar.gz
tar_candidates = list(CBIO_DIR.glob("*.tar.gz"))
assert tar_candidates, f"No .tar.gz found in {CBIO_DIR}"

# Prefer coad_tcga_gdc
target_tar = next((t for t in tar_candidates if "coad" in t.name.lower()), tar_candidates[0])
print(f"Loading: {target_tar.name}")
cbio = read_cbio_tar(target_tar)

sample_df = cbio["sample"]
patient_df = cbio.get("patient", None)

tmb_cols = [c for c in sample_df.columns if "TMB" in c.upper()]
print(f"Sample rows: {len(sample_df):,}  | TMB columns: {tmb_cols}")
sample_df.head(3)


SyntaxError: unterminated string literal (detected at line 12) (1131011433.py, line 12)

In [ ]:
# Normalise column names and build label dataframe
TMB_COL = tmb_cols[0]  # e.g. TMB_NONSYNONYMOUS

label_df = (
    sample_df
    .with_columns(pl.col("PATIENT_ID").str.to_uppercase())
    .with_columns(pl.col(TMB_COL).cast(pl.Float64, strict=False).alias("TMB"))
    .drop_nulls(subset=["TMB"])
    .with_columns([
        # Binary label: TMB-H (1) vs rest (0) — paper Section III-A
        pl.when(pl.col("TMB") >= TMB_HIGH).then(pl.lit(1)).otherwise(pl.lit(0)).alias("binary_label"),
        # Ternary label: 2=High, 1=Medium, 0=Low
        pl.when(pl.col("TMB") >= TMB_HIGH).then(pl.lit(2))
          .when(pl.col("TMB") >= TMB_LOW).then(pl.lit(1))
          .otherwise(pl.lit(0)).alias("ternary_label"),
    ])
    .select(["PATIENT_ID", "TMB", "binary_label", "ternary_label"])
    .rename({"PATIENT_ID": "patient_id"})
)

print(f"Labelled patients: {len(label_df):,}")
print("Binary distribution:")
print(label_df["binary_label"].value_counts())
print("Ternary distribution:")
print(label_df["ternary_label"].value_counts())


## § 4 — Load Pre-Tiled Patches & Build Manifest

Each subdirectory of  is named after a TCGA Case ID.
We walk the directory to build a tile manifest DataFrame.


In [ ]:
TILE_MANIFEST_PATH = DATA_DIR / "cellmorphnet_tile_manifest.csv"

if TILE_MANIFEST_PATH.exists():
    tile_df = pl.read_csv(str(TILE_MANIFEST_PATH))
    print(f"Loaded cached manifest: {len(tile_df):,} tiles")
else:
    records = []
    case_dirs = [d for d in TILES_DIR.iterdir() if d.is_dir()]
    print(f"Found {len(case_dirs)} case directories")
    for case_dir in tqdm(case_dirs, desc="Scanning tiles"):
        patient_id = case_dir.name.upper()
        for img_path in case_dir.glob("*.png"):
            records.append({"path": str(img_path), "patient_id": patient_id})
        for img_path in case_dir.glob("*.jpg"):
            records.append({"path": str(img_path), "patient_id": patient_id})
    tile_df = pl.DataFrame(records) if records else pl.DataFrame({"path": [], "patient_id": []})
    tile_df.write_csv(str(TILE_MANIFEST_PATH))
    print(f"Built manifest: {len(tile_df):,} tiles from {tile_df['patient_id'].n_unique()} cases")

tile_df.head(3)


In [ ]:
# Join tile manifest with TMB labels on patient_id
# Normalise both patient IDs to uppercase for reliable matching
tile_df = tile_df.with_columns(pl.col("patient_id").str.to_uppercase())
label_df = label_df.with_columns(pl.col("patient_id").str.to_uppercase())

tiled_df = tile_df.join(label_df, on="patient_id", how="inner")

print(f"After join: {len(tiled_df):,} tiles  |  "
      f"{tiled_df['patient_id'].n_unique()} unique patients")
print("Binary label distribution in tile dataset:")
print(tiled_df["binary_label"].value_counts())


## § 5 — Exploratory Data Analysis


In [ ]:
# --- 5.1  TMB Distribution across unique patients ---
unique_pts = label_df.unique("patient_id")
tmb_vals   = unique_pts["TMB"].to_list()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
axes[0].hist(tmb_vals, bins=40, color="slateblue", edgecolor="white")
axes[0].axvline(TMB_HIGH, color="crimson", lw=2, ls="--", label=f"TMB-H ≥ {TMB_HIGH}")
axes[0].axvline(TMB_LOW,  color="orange",  lw=2, ls=":", label=f"TMB-L < {TMB_LOW}")
axes[0].set(xlabel="TMB (mut/Mb)", ylabel="# Patients", title="TMB Distribution")
axes[0].legend()

# Log histogram
axes[1].hist(tmb_vals, bins=40, color="steelblue", edgecolor="white", log=True)
axes[1].axvline(TMB_HIGH, color="crimson", lw=2, ls="--")
axes[1].axvline(TMB_LOW,  color="orange",  lw=2, ls=":")
axes[1].set(xlabel="TMB", ylabel="Count (log)", title="TMB Distribution (log scale)")

# Binary class balance
bc = unique_pts["binary_label"].value_counts().sort("binary_label")
labels_b = ["TMB-L/M", "TMB-H"]
counts_b = bc["count"].to_list()
bars = axes[2].bar(labels_b, counts_b, color=["#3498db", "#e74c3c"], width=0.5)
for bar, cnt in zip(bars, counts_b):
    axes[2].text(bar.get_x() + bar.get_width()/2, cnt+0.3, str(cnt), ha="center", fontweight="bold")
axes[2].set(title="Binary Class Balance")

plt.tight_layout()
plt.savefig(str(SPLITS_DIR / "eda_tmb_distribution.png"), dpi=120, bbox_inches="tight")
plt.show()
print(f"Mean TMB: {np.mean(tmb_vals):.2f} | Median: {np.median(tmb_vals):.2f} | Max: {max(tmb_vals):.2f}")


In [ ]:
# --- 5.2  Tiles-per-patient statistics ---
if len(tiled_df) > 0:
    tile_counts = (
        tiled_df.group_by("patient_id")
        .agg([
            pl.len().alias("n_tiles"),
            pl.col("binary_label").first().alias("label")
        ])
    )
    tile_counts_pd = tile_counts.to_pandas()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, cls, col in zip(axes, [0, 1], ["#3498db", "#e74c3c"]):
        subset = tile_counts_pd[tile_counts_pd["label"] == cls]["n_tiles"]
        ax.hist(subset, bins=20, color=col, edgecolor="white")
        ax.set(title=f"Tiles per patient — {'TMB-H' if cls==1 else 'TMB-L/M'}",
               xlabel="# Tiles", ylabel="# Patients")
        ax.text(0.97, 0.97, f"Median={subset.median():.0f}", transform=ax.transAxes,
                ha="right", va="top")

    plt.tight_layout()
    plt.savefig(str(SPLITS_DIR / "eda_tile_counts.png"), dpi=120, bbox_inches="tight")
    plt.show()
    print(tile_counts.describe())
else:
    print("[INFO] No tiles found — run tile extraction pipeline first (see Phase-2 notebook § 8).")


In [ ]:
# --- 5.3  Sample tile visualisation (4 × 4 grid) ---
if len(tiled_df) > 0:
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, row in zip(axes.ravel(), tiled_df.sample(16, seed=42).iter_rows(named=True)):
        img = Image.open(row["path"]).resize((224, 224)).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{'TMB-H' if row['binary_label']==1 else 'TMB-L'}
{row['patient_id'][:12]}", fontsize=6)
        ax.axis("off")
    plt.suptitle("Sample Histopathology Tiles — 224×224 px", y=1.01)
    plt.tight_layout()
    plt.savefig(str(SPLITS_DIR / "eda_sample_tiles.png"), dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("[INFO] No tiles to visualise.")


## § 6 — Feature Engineering & Class Imbalance Handling

**Class weights** follow the paper (Section III-A): inverse-frequency weighting.
For binary classification the paper reports ~2.48× higher weight for low-TMB samples.


In [ ]:
# ── Class weight computation (inverse frequency — paper Sec. III-A) ──────
if len(tiled_df) > 0:
    n_h = int(tiled_df.filter(pl.col("binary_label") == 1).shape[0])
    n_l = int(tiled_df.filter(pl.col("binary_label") == 0).shape[0])
    tot = n_h + n_l
    # w_c = N / (n_classes * n_c)
    w_high = tot / (2 * n_h) if n_h > 0 else 1.0
    w_low  = tot / (2 * n_l) if n_l > 0 else 1.0
    CLASS_WEIGHTS = {0: w_low, 1: w_high}
    CLASS_WEIGHTS_TENSOR = [w_low, w_high]  # index 0 = TMB-L, 1 = TMB-H
    print(f"TMB-H tiles: {n_h:,}  |  TMB-L/M tiles: {n_l:,}")
    print(f"Class weights: {{0: {w_low:.3f}, 1: {w_high:.3f}}}")
    print(f"Imbalance ratio: {w_low/w_high:.2f}× (paper reports ~2.48×)")
else:
    print("[INFO] Placeholder — set equal weights until tiles are available.")
    CLASS_WEIGHTS = {0: 1.0, 1: 1.0}
    CLASS_WEIGHTS_TENSOR = [1.0, 1.0]


In [ ]:
# ── Texture & stain features per patient (for PDP/ICE in model notebook) ─
feature_records = []

if len(tiled_df) > 0:
    sample_tiles = tiled_df.group_by("patient_id").head(10)  # up to 10 tiles/patient
    for row in tqdm(sample_tiles.iter_rows(named=True), desc="Computing features",
                     total=len(sample_tiles)):
        try:
            img = np.array(Image.open(row["path"]).resize((224, 224)).convert("RGB")).astype(np.float32)
            h_channel = img[:, :, 0]
            feature_records.append({
                "patient_id":       row["patient_id"],
                "binary_label":     row["binary_label"],
                "TMB":              row["TMB"],
                "mean_r":           float(img[:, :, 0].mean()),
                "mean_g":           float(img[:, :, 1].mean()),
                "mean_b":           float(img[:, :, 2].mean()),
                "std_r":            float(img[:, :, 0].std()),
                "std_g":            float(img[:, :, 1].std()),
                "std_b":            float(img[:, :, 2].std()),
                "haem_intensity":   float((img[:, :, 0] - img[:, :, 1]).mean()),  # H channel proxy
                "eosin_intensity":  float((img[:, :, 1] - img[:, :, 2]).mean()),  # E channel proxy
            })
        except Exception as e:
            pass

    features_df = pd.DataFrame(feature_records)
    features_df.to_csv(str(SPLITS_DIR / "tile_features.csv"), index=False)
    print(f"Computed features for {len(features_df):,} tile samples")
else:
    print("[INFO] No tiles — features will be synthetic in model notebook.")
    features_df = pd.DataFrame()


## § 7 — Stratified Train / Validation / Test Splits

Split at **patient level** (never at tile level) to prevent data leakage.
Stratify by binary TMB class. Paper uses 80/20 train/test; we add a 10% validation slice.


In [ ]:
SEED = 42

def make_patient_splits(label_df: pl.DataFrame, seed: int = 42):
    """
    Returns train_pts, val_pts, test_pts — lists of patient_id strings.
    Stratified by binary_label. Paper 80/20 train/test; we use 70/10/20.
    """
    pts_h = label_df.filter(pl.col("binary_label") == 1)["patient_id"].to_list()
    pts_l = label_df.filter(pl.col("binary_label") == 0)["patient_id"].to_list()

    tr_h, tmp_h = train_test_split(pts_h, test_size=0.30, random_state=seed)
    va_h, te_h  = train_test_split(tmp_h, test_size=0.67, random_state=seed)
    tr_l, tmp_l = train_test_split(pts_l, test_size=0.30, random_state=seed)
    va_l, te_l  = train_test_split(tmp_l, test_size=0.67, random_state=seed)

    return tr_h + tr_l, va_h + va_l, te_h + te_l


if len(label_df) > 0:
    train_pts, val_pts, test_pts = make_patient_splits(label_df, seed=SEED)
    print(f"Train patients: {len(train_pts):4d}  "
          f"Val: {len(val_pts):4d}  Test: {len(test_pts):4d}")

    splits = {"train": train_pts, "val": val_pts, "test": test_pts}
    with open(str(SPLITS_DIR / "patient_splits.json"), "w") as f:
        json.dump(splits, f, indent=2)
    print("Saved → members/youssef/experiments/CellMorphNet/patient_splits.json")

    # Verify label proportions in each split
    for split_name, pts in splits.items():
        sub = label_df.filter(pl.col("patient_id").is_in(pts))
        nh = sub.filter(pl.col("binary_label") == 1).shape[0]
        nl = sub.filter(pl.col("binary_label") == 0).shape[0]
        print(f"  {split_name:5s} — TMB-H: {nh}  TMB-L/M: {nl}  ratio: {nh/(nh+nl)*100:.1f}%")
else:
    print("[WARNING] label_df empty — cannot create splits.")


In [ ]:
# ── Also save class weights ─────────────────────────────────────────────
meta = {
    "class_weights": CLASS_WEIGHTS_TENSOR,
    "tmb_high_threshold": TMB_HIGH,
    "tmb_low_threshold":  TMB_LOW,
    "seed": SEED,
    "tile_manifest": str(TILE_MANIFEST_PATH),
}
with open(str(SPLITS_DIR / "preprocessing_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved → members/youssef/experiments/CellMorphNet/preprocessing_meta.json")
print("
--- Preprocessing complete ---")
print(f"All outputs saved to: {SPLITS_DIR}")
